#### ***Causal Attention Mechansim***

In [176]:
import torch


inputs = torch.tensor([
    [0.8, 0.1, 0.2],   #← Your
    [1.0, 0.7, 0.8],   #← Journey
    [0.7, 1.0, 0.3],   #← Starts
    [0.2, 0.2, 0.6],   #← with
    [0.3, 0.4, 0.9],   #← one
    [0.8, 0.9, 1.0]    #← step
])

In [177]:
x_2 = inputs[1]

d_in = inputs.shape[1]
d_out = 2
d_in

3

In [178]:
import torch.nn as nn

class SimpleAttention_V2(nn.Module):

    def __init__(self,d_in,s_out,qvk_bias = False):
        super().__init__()
        self.W_query = nn.Linear(d_in,d_out,bias = qvk_bias)
        self.W_key = nn.Linear(d_in,d_out,bias = qvk_bias)
        self.W_value = nn.Linear(d_in,d_out,bias = qvk_bias)

    def forward(self,x):
        keys = self.W_key(x)
        queries = self.W_query(x) 
        values = self.W_value(x)


        attention_scores = queries@keys.T
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5,dim= -1)
        context_vectors = attention_weights@values

        return context_vectors

In [179]:
inputs

tensor([[0.8000, 0.1000, 0.2000],
        [1.0000, 0.7000, 0.8000],
        [0.7000, 1.0000, 0.3000],
        [0.2000, 0.2000, 0.6000],
        [0.3000, 0.4000, 0.9000],
        [0.8000, 0.9000, 1.0000]])

In [180]:
torch.manual_seed(123)
sa_v2 = SimpleAttention_V2(d_in,d_out)
queries = sa_v2.W_query(inputs)
keys= sa_v2.W_key(inputs)

attention_scores =queries@keys.T

attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim = -1)
print(attention_weights)

tensor([[0.1593, 0.1774, 0.1660, 0.1555, 0.1633, 0.1785],
        [0.1521, 0.1844, 0.1682, 0.1472, 0.1602, 0.1878],
        [0.1576, 0.1718, 0.1735, 0.1582, 0.1634, 0.1755],
        [0.1602, 0.1763, 0.1658, 0.1567, 0.1637, 0.1772],
        [0.1568, 0.1805, 0.1661, 0.1522, 0.1622, 0.1822],
        [0.1516, 0.1839, 0.1693, 0.1472, 0.1601, 0.1877]],
       grad_fn=<SoftmaxBackward0>)


In [181]:
attention_scores.shape

torch.Size([6, 6])

In [182]:
torch.ones(attention_scores.shape)

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [183]:
### using the Pytorch trill function we can create/make the values in above diagonal's are zero
context_length = attention_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length,context_length))
mask_simple

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [184]:
attention_weights

tensor([[0.1593, 0.1774, 0.1660, 0.1555, 0.1633, 0.1785],
        [0.1521, 0.1844, 0.1682, 0.1472, 0.1602, 0.1878],
        [0.1576, 0.1718, 0.1735, 0.1582, 0.1634, 0.1755],
        [0.1602, 0.1763, 0.1658, 0.1567, 0.1637, 0.1772],
        [0.1568, 0.1805, 0.1661, 0.1522, 0.1622, 0.1822],
        [0.1516, 0.1839, 0.1693, 0.1472, 0.1601, 0.1877]],
       grad_fn=<SoftmaxBackward0>)

In [185]:
### let's multiplication with attn_weights
masked_simple = attention_weights*mask_simple
masked_simple

tensor([[0.1593, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1521, 0.1844, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1576, 0.1718, 0.1735, 0.0000, 0.0000, 0.0000],
        [0.1602, 0.1763, 0.1658, 0.1567, 0.0000, 0.0000],
        [0.1568, 0.1805, 0.1661, 0.1522, 0.1622, 0.0000],
        [0.1516, 0.1839, 0.1693, 0.1472, 0.1601, 0.1877]],
       grad_fn=<MulBackward0>)

In [186]:
### renormalize the masked simple to sum up to 1 on each row

In [187]:
rows_sum = masked_simple.sum(dim=1,keepdim=True)
masked_simple_norm = masked_simple/rows_sum
masked_simple_norm

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4520, 0.5480, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3133, 0.3417, 0.3450, 0.0000, 0.0000, 0.0000],
        [0.2431, 0.2675, 0.2516, 0.2378, 0.0000, 0.0000],
        [0.1917, 0.2207, 0.2031, 0.1861, 0.1983, 0.0000],
        [0.1516, 0.1839, 0.1693, 0.1472, 0.1601, 0.1877]],
       grad_fn=<DivBackward0>)

In [188]:
### the above steps may lead to data leakage issue because we are using original attention weights,
### making them masked to above diagonal's are zero

### more efficient way
##1.Attention Scores
###2.Upper Triangular mask infinity
##3.softmax

In [189]:
attention_scores

tensor([[0.1593, 0.3115, 0.2174, 0.1254, 0.1948, 0.3204],
        [0.2542, 0.5267, 0.3966, 0.2077, 0.3277, 0.5522],
        [0.0562, 0.1787, 0.1924, 0.0618, 0.1079, 0.2085],
        [0.1439, 0.2791, 0.1925, 0.1127, 0.1747, 0.2862],
        [0.2036, 0.4027, 0.2854, 0.1614, 0.2516, 0.4157],
        [0.2439, 0.5170, 0.4001, 0.2022, 0.3211, 0.5460]],
       grad_fn=<MmBackward0>)

In [190]:
torch.ones(context_length,context_length)

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [191]:
torch.triu(torch.ones(context_length,context_length))
## diagonal all ones

tensor([[1., 1., 1., 1., 1., 1.],
        [0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.]])

In [192]:
torch.triu(torch.ones(context_length,context_length),diagonal=1)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])

In [193]:
mask = torch.triu(torch.ones(context_length,context_length),diagonal=1)

masked = attention_scores.masked_fill(mask.bool(),-torch.inf)
# means mask.bool --> true replace that values -inf.
masked

tensor([[0.1593,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.2542, 0.5267,   -inf,   -inf,   -inf,   -inf],
        [0.0562, 0.1787, 0.1924,   -inf,   -inf,   -inf],
        [0.1439, 0.2791, 0.1925, 0.1127,   -inf,   -inf],
        [0.2036, 0.4027, 0.2854, 0.1614, 0.2516,   -inf],
        [0.2439, 0.5170, 0.4001, 0.2022, 0.3211, 0.5460]],
       grad_fn=<MaskedFillBackward0>)

In [194]:
masked_attn_weights = torch.softmax(masked/keys.shape[-1]**0.5,dim=1)
masked_attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4520, 0.5480, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3133, 0.3417, 0.3450, 0.0000, 0.0000, 0.0000],
        [0.2431, 0.2675, 0.2516, 0.2378, 0.0000, 0.0000],
        [0.1917, 0.2207, 0.2031, 0.1861, 0.1983, 0.0000],
        [0.1516, 0.1839, 0.1693, 0.1472, 0.1601, 0.1877]],
       grad_fn=<SoftmaxBackward0>)

In [195]:
### Masking additional attention weights with DropOut

In [196]:
examples = torch.ones(6,6)
examples

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [197]:
dropout = torch.nn.Dropout(0.5) # 50 %
dropout

Dropout(p=0.5, inplace=False)

In [198]:
torch.manual_seed(123)
examples = torch.ones(6,6)
dropout = torch.nn.Dropout(0.5) # 50 %
dropout(examples)

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])

In [199]:
## when we set dropout to 0.5, half of elements in the matrix set to zero. 
## remaining elements are scaled up by factor 1/0.5 = 2.
# let's apply dropout to the Attention Matrix itself

In [200]:
dropout(masked_attn_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6833, 0.6900, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.5032, 0.4756, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3967, 0.0000],
        [0.3033, 0.3679, 0.0000, 0.2945, 0.3203, 0.3755]],
       grad_fn=<MulBackward0>)

In [201]:
## Create a batch which can handle the multiple inputs
batch = torch.stack((inputs,inputs),dim=0)
batch.shape

torch.Size([2, 6, 3])

In [202]:
#The result is 3D-tensor matrix with 2 inputs texts with 6 tokens,where each token is a 3-Dimensional


In [203]:
import torch.nn as nn

class CausalAttention(nn.Module):

    def __init__(self,d_in,d_out,context_length,dropout,qvk_bias = False):
        super().__init__()
        self.W_query = nn.Linear(d_in,d_out,bias = qvk_bias)
        self.W_key = nn.Linear(d_in,d_out,bias = qvk_bias)
        self.W_value = nn.Linear(d_in,d_out,bias = qvk_bias)
        self.drop_out = nn.Dropout(dropout)
        self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length),diagonal=1))

    def forward(self,x):
        b,num_tokens,d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x) 
        values = self.W_value(x)


        attention_scores = queries@keys.transpose(1,2)
        attention_scores.masked_fill_(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5,dim= -1)
        attention_weights = self.drop_out(attention_weights)
        context_vectors = attention_weights@values

        return context_vectors

In [204]:
d_in,d_out

(3, 2)

In [205]:
context_length = batch.shape[1]
context_length

6

In [206]:
ca = CausalAttention(d_in,d_out,context_length,0.0)
context_vector = ca(batch)
print("Context Vectors Shape:",context_vector.shape)

Context Vectors Shape: torch.Size([2, 6, 2])


In [207]:
print(context_vector)

tensor([[[-0.3878,  0.3887],
         [-0.5494,  0.6065],
         [-0.4860,  0.5785],
         [-0.4597,  0.5435],
         [-0.4749,  0.5610],
         [-0.5139,  0.6119]],

        [[-0.3878,  0.3887],
         [-0.5494,  0.6065],
         [-0.4860,  0.5785],
         [-0.4597,  0.5435],
         [-0.4749,  0.5610],
         [-0.5139,  0.6119]]], grad_fn=<UnsafeViewBackward0>)
